# Decision Intelligence Assistant

This notebook covers data preparation, weak labeling, feature engineering, model comparison, and artifact export for the project.

## 1. Goals

- Build a representative 10k-row sample from the Twitter support dataset.
- Create a transparent weak-labeling rule for ticket priority.
- Compare TF-IDF, engineered, and combined features.
- Select a deployable ML baseline and export its artifacts.
- Collect examples for the final written analysis.

## 2. Imports

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import re
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
TARGET_SAMPLE_SIZE = 10_000

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_raw_dir = project_root / "data" / "raw"
data_sample_dir = project_root / "data" / "sample"
artifacts_dir = project_root / "artifacts"
backend_path = project_root / "backend"
if str(backend_path) not in sys.path:
    sys.path.append(str(backend_path))

data_sample_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print(f"Project root: {project_root}")
print(f"Raw data dir: {data_raw_dir}")
print(f"Sample output dir: {data_sample_dir}")


Project root: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant
Raw data dir: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\raw
Sample output dir: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\sample


In [2]:
from pathlib import Path
import shutil

try:
    import kagglehub
except ImportError:
    kagglehub = None

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

if kagglehub is None:
    print("kagglehub is not installed. Place twcs.csv in data/raw manually.")
else:
    download_path = Path(
        kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
    )
    print("Downloaded to:", download_path)

    csv_files = list(download_path.rglob("*.csv"))
    print("CSV files found:", [p.name for p in csv_files])

    preferred = next((p for p in csv_files if p.name == "twcs.csv"), None)
    source_csv = preferred or next((p for p in csv_files if p.name == "sample.csv"), None)
    if source_csv is None:
        raise FileNotFoundError("No CSV file was found in the downloaded dataset.")

    target_csv = raw_dir / source_csv.name
    shutil.copy2(source_csv, target_csv)
    print("Copied dataset to:", target_csv)
    if source_csv.name != "twcs.csv":
        print("Warning: using sample.csv fallback, not the full twcs.csv dataset.")


kagglehub is not installed. Place twcs.csv in data/raw manually.


## 3. Load Raw Dataset

In [3]:
def parse_linked_ids(raw_value: Any) -> list[str]:
    """Parse linked tweet identifiers from a raw cell value."""
    if pd.isna(raw_value):
        return []

    raw_text = str(raw_value).strip()
    if not raw_text:
        return []

    return [token for token in re.split(r"[\s,\[\]]+", raw_text) if token]


def to_bool_flag(raw_value: Any) -> bool:
    """Coerce the inbound column to a real boolean flag."""
    if isinstance(raw_value, bool):
        return raw_value
    if pd.isna(raw_value):
        return False

    normalized = str(raw_value).strip().lower()
    return normalized in {"true", "t", "1", "yes"}


def load_raw_dataset(raw_dir: Path) -> tuple[pd.DataFrame, Path]:
    """Load the first supported dataset found in data/raw."""
    csv_candidates = sorted(raw_dir.glob("*.csv"))
    parquet_candidates = sorted(raw_dir.glob("*.parquet"))
    preferred_csv = [path for path in csv_candidates if path.name == "twcs.csv"]
    fallback_csv = [path for path in csv_candidates if path.name != "twcs.csv"]
    candidates = preferred_csv + fallback_csv + parquet_candidates

    if not candidates:
        raise FileNotFoundError(
            "No raw dataset was found in data/raw. Place the Kaggle Twitter support "
            "CSV or Parquet file there before running this notebook."
        )

    dataset_path = candidates[0]
    if dataset_path.suffix == ".csv":
        dataframe = pd.read_csv(dataset_path)
    else:
        dataframe = pd.read_parquet(dataset_path)

    return dataframe, dataset_path


raw_df, dataset_path = load_raw_dataset(data_raw_dir)
print(f"Loaded dataset: {dataset_path.name}")
print(f"Raw shape: {raw_df.shape}")

required_columns = {"tweet_id", "author_id", "inbound", "text"}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = raw_df.copy()
df["tweet_id"] = df["tweet_id"].astype(str)
df["author_id"] = df["author_id"].astype(str)
df["text"] = df["text"].fillna("").astype(str)
df["inbound_flag"] = df["inbound"].apply(to_bool_flag)

def is_likely_brand(author_id: str) -> bool:
    """Identify brand/support handles, which are text names in this dataset."""
    return not str(author_id).strip().isdigit()


def infer_brand_hint(row: pd.Series, tweet_to_author: dict[str, str]) -> str:
    """Infer the support brand from linked response/context tweets."""
    linked_ids = []
    if "response_tweet_id" in row.index:
        linked_ids.extend(parse_linked_ids(row.get("response_tweet_id")))
    if "in_response_to_tweet_id" in row.index:
        linked_ids.extend(parse_linked_ids(row.get("in_response_to_tweet_id")))

    linked_authors = [tweet_to_author[tweet_id] for tweet_id in linked_ids if tweet_id in tweet_to_author]
    return next((author for author in linked_authors if is_likely_brand(author)), "unknown")


tweet_to_author = df.set_index("tweet_id")["author_id"].to_dict()
df["brand_hint"] = df.apply(lambda row: infer_brand_hint(row, tweet_to_author), axis=1)

customer_tickets = df.loc[df["inbound_flag"]].copy()
customer_tickets["text"] = customer_tickets["text"].str.strip()
customer_tickets = customer_tickets.loc[customer_tickets["text"].ne("")].copy()
customer_tickets["normalized_text"] = (
    customer_tickets["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
customer_tickets = customer_tickets.drop_duplicates(subset=["tweet_id"])
customer_tickets = customer_tickets.drop_duplicates(subset=["normalized_text"])

customer_tickets["text_length"] = customer_tickets["text"].str.len()
customer_tickets["word_count"] = customer_tickets["text"].str.split().str.len()

print(f"Customer-ticket shape after cleaning: {customer_tickets.shape}")
display(customer_tickets.head())


Loaded dataset: twcs.csv
Raw shape: (2811774, 7)


Customer-ticket shape after cleaning: (1508534, 12)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,inbound_flag,brand_hint,normalized_text,text_length,word_count
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0,True,unknown,@sprintcare and how do you propose we do that,45,9
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messages and no one is responding as usual,1,4.0,True,sprintcare,@sprintcare i have sent several private messages and no one is responding as usual,82,14
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0,True,sprintcare,@sprintcare i did.,18,3
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN,True,sprintcare,@sprintcare is the worst customer service,41,6
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯,"11,13,14",15.0,True,sprintcare,@sprintcare you gonna magically change your connectivity for me and my whole family ? 🤥 💯,89,16


## 4. Build 10k Representative Sample

In [4]:
def build_representative_sample(
    dataframe: pd.DataFrame,
    target_size: int,
    random_state: int = 42,
) -> pd.DataFrame:
    """Create a reproducible ticket sample that preserves brand variety."""
    if dataframe.empty:
        raise ValueError("Cannot sample from an empty dataframe.")

    sample_size = min(target_size, len(dataframe))
    working_df = dataframe.copy()
    working_df["brand_hint"] = working_df["brand_hint"].fillna("unknown")

    group_sizes = working_df["brand_hint"].value_counts()
    exact_allocations = group_sizes / group_sizes.sum() * sample_size
    allocations = np.floor(exact_allocations).astype(int)
    allocations[allocations == 0] = 1

    while allocations.sum() > sample_size:
        largest_group = allocations.sort_values(ascending=False).index[0]
        if allocations[largest_group] > 1:
            allocations[largest_group] -= 1
        else:
            break

    while allocations.sum() < sample_size:
        remainders = exact_allocations - allocations
        next_group = remainders.sort_values(ascending=False).index[0]
        allocations[next_group] += 1

    samples = []
    for brand_hint, group_df in working_df.groupby("brand_hint", sort=False):
        requested_n = min(allocations.get(brand_hint, 0), len(group_df))
        if requested_n == 0:
            continue
        sampled_group = group_df.sample(n=requested_n, random_state=random_state)
        samples.append(sampled_group)

    sample_df = pd.concat(samples, ignore_index=True)
    if len(sample_df) > sample_size:
        sample_df = sample_df.sample(n=sample_size, random_state=random_state)

    return sample_df.sort_values("tweet_id").reset_index(drop=True)


sample_df = build_representative_sample(
    dataframe=customer_tickets,
    target_size=TARGET_SAMPLE_SIZE,
    random_state=SEED,
)

sample_output_path = data_sample_dir / "customer_support_sample.csv"
sample_df.to_csv(sample_output_path, index=False)

print(f"Saved representative sample to: {sample_output_path}")
print(f"Sample shape: {sample_df.shape}")
print(
    "Brand coverage in sample:",
    sample_df["brand_hint"].nunique(),
)

sample_summary = pd.DataFrame(
    {
        "full_dataset_count": customer_tickets["brand_hint"].value_counts(),
        "sample_count": sample_df["brand_hint"].value_counts(),
    }
).fillna(0).astype(int)
sample_summary["sample_share"] = (
    sample_summary["sample_count"] / sample_summary["sample_count"].sum()
).round(4)

display(sample_summary.head(15))
display(sample_df[["tweet_id", "author_id", "brand_hint", "text"]].head(10))


Saved representative sample to: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\data\sample\customer_support_sample.csv
Sample shape: (10000, 12)
Brand coverage in sample: 109


,full_dataset_count,sample_count,sample_share
brand_hint,,,
ATT,2607,17,0.0017
ATVIAssist,16699,111,0.0111
AWSSupport,1018,7,0.0007
AdobeCare,8175,54,0.0054
AirAsiaSupport,10369,69,0.0069
AirbnbHelp,8262,55,0.0055
AlaskaAir,7280,48,0.0048
AldiUK,7604,50,0.0050
AmazonHelp,152539,1011,0.1011


,tweet_id,author_id,brand_hint,text
0,1000186,356792,AppleSupport,@115858 @AppleSupport y’all need to fix this fr I️ shouldn’t be having this petty ass problem
1,1000225,356801,AppleSupport,I need @115858 to fix this fucking issue 👉🏼 I️
2,1000677,167119,Uber_Support,@Uber_Support u all r getting worst day by day. Terrible experience everytime..fed up of u all
3,1001005,161904,AmazonHelp,@AmazonHelp I preordered amazon echo but want to go for amazon plus now. Anyway to reach out to seller? I want another invitation.
4,100102,137934,Uber_Support,"@Uber_Support Ok, sent a DM about 10 mins ago. Looking forward to hearing from you. 🙂"
5,1001122,357008,AppleSupport,@AppleSupport My carrier BSNL allows hotspot. Please tell me what should I do now?
6,1001438,357075,unknown,"@Tesco Tesco selling old chickens sent back from Aldi https://t.co/vtxsoWxs0v\nRetweet people, spread the word"
7,1001601,357112,SW_Help,@SW_Help Do you no longer travel direct from Exeter St Davids to London Waterloo?
8,1001655,228355,unknown,@ArgosHelpers @115830 Your in store system is designed for the lowest common denominator yet it still evaded my understanding. And I'm like really clever.
9,1001663,228355,unknown,@ArgosHelpers @357121 @115830 Instead they are fundamental to even seeing about availability. The iPads don't give info and send confirmation too late. Waste of time.


## 5. Weak Labeling Rule

In [5]:
from backend.app.services.features import extract_features, weak_label_priority

sample_df["priority_label"] = sample_df["text"].apply(weak_label_priority)
label_distribution = sample_df["priority_label"].value_counts().rename_axis("priority").reset_index(name="count")
label_distribution["share"] = (label_distribution["count"] / len(sample_df)).round(3)

print("Weak-label distribution")
display(label_distribution)

display(
    sample_df[["tweet_id", "brand_hint", "text", "priority_label"]]
    .sort_values("priority_label")
    .head(12)
)


Weak-label distribution


,priority,count,share
0,low,7370,0.737
1,normal,2082,0.208
2,high,511,0.051
3,urgent,37,0.004


,tweet_id,brand_hint,text,priority_label
7932,440985,AmazonHelp,"@115850 pls forward request sent to TCL tv installation for my order 406-0519449-9058739,TCL support asking for it, pls help ASAP",high
9199,783514,British_Airways,@British_Airways Why oh why can I not actually login anymore on my account? I have a flight in 12 days and I can't get in to make changes.,high
4964,2315329,Ask_Spectrum,@Ask_Spectrum Good evening. Is there an Internet outage on the #uppereastside of #Manhattan ? I have no Internet service since midnight.,high
6188,2645195,AskPayPal,@AskPayPal Just sent DM regarding a refund if someone could help asap I'd appreciate it. Tried calling but was on hold for 43 minutes,high
6543,2746544,unknown,"@BofA_Help Yes and he was denied. I am disgusted and frustrated. How dare BOA deny returning our money when the banks were bailed out during the recession, (while we lost our h...",high
8705,652739,British_Airways,@British_Airways calling regarding a cancelled flight for tomorrow - 20 minutes waiting on your phone line so far!!!,high
1527,1398539,unknown,@Uber_Support She was logged in through her phone so she cannot access her account..I need a contact number.,high
6803,2816120,unknown,Is anyone else with @115725 experiencing DREADFUL service lately?! Full LTE and yet my apps and webpages can’t load. How is that possible? #VZW #terrible #whatdoipayfor,high
1662,1431900,BofA_Help,Hey @116035 what is the deal with your customer service line? You can't afford to update that automated system you set up in the 80s?,high
1523,1397334,unknown,@AppleSupport The infamous “ninja update when I’m sleeping” it crashed my phone. The 6s can’t handle current update. You can help me by discounting an X,high


## 6. Feature Engineering

In [6]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

feature_rows = [extract_features(text).__dict__ for text in sample_df["text"]]
engineered_df = pd.DataFrame(feature_rows)

vectorizer = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    min_df=1,
    stop_words="english",
)
tfidf_matrix = vectorizer.fit_transform(sample_df["text"])
engineered_matrix = engineered_df.to_numpy(dtype=float)
combined_matrix = hstack([tfidf_matrix, engineered_matrix])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(sample_df["priority_label"])

print("TF-IDF shape:", tfidf_matrix.shape)
print("Engineered feature shape:", engineered_matrix.shape)
display(engineered_df.describe().T)


TF-IDF shape: (10000, 20000)
Engineered feature shape: (10000, 10)


,count,mean,std,min,25%,50%,75%,max
char_count,10000.0,111.441200,56.386769,7.0,69.000000,110.0000,140.0,424.0
word_count,10000.0,19.061700,10.420468,0.0,11.000000,19.0000,25.0,61.0
exclamation_count,10000.0,0.264900,0.828853,0.0,0.000000,0.0000,0.0,18.0
question_count,10000.0,0.353900,0.713236,0.0,0.000000,0.0000,1.0,10.0
uppercase_ratio,10000.0,0.084345,0.092727,0.0,0.038961,0.0625,0.1,1.0
urgent_keyword_count,10000.0,0.189900,0.481104,0.0,0.000000,0.0000,0.0,5.0
high_impact_keyword_count,10000.0,0.251700,0.548067,0.0,0.000000,0.0000,0.0,6.0
negative_keyword_count,10000.0,0.064700,0.266685,0.0,0.000000,0.0000,0.0,4.0
has_url,10000.0,0.140100,0.347108,0.0,0.000000,0.0000,0.0,1.0
has_mention,10000.0,0.963100,0.188526,0.0,1.000000,1.0000,1.0,1.0


## 7. Model Comparison

In [7]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

feature_sets = {
    "tfidf_only": tfidf_matrix,
    "engineered_only": engineered_matrix,
    "tfidf_engineered": combined_matrix,
}
models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced"),
    "LinearSVC": LinearSVC(max_iter=20000, class_weight="balanced"),
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=1,
    ),
}

results = []
trained_models = {}

class_counts = pd.Series(y).value_counts()
stratify_labels = y if class_counts.min() >= 2 else None
if stratify_labels is None:
    print(
        "Warning: at least one class has fewer than 2 rows, so this smoke-test "
        "run uses a regular split. Use twcs.csv/10k sample for final metrics."
    )

for feature_name, features in feature_sets.items():
    x_train, x_test, y_train, y_test = train_test_split(
        features,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=stratify_labels,
    )
    for model_name, model in models.items():
        start = time.perf_counter()
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)
        latency_ms = ((time.perf_counter() - start) / max(len(y_test), 1)) * 1000
        macro_f1 = f1_score(y_test, predictions, average="macro")
        results.append(
            {
                "feature_set": feature_name,
                "model": model_name,
                "accuracy": accuracy_score(y_test, predictions),
                "macro_f1": macro_f1,
                "latency_ms_per_prediction": latency_ms,
            }
        )
        trained_models[(feature_name, model_name)] = (model, x_test, y_test, predictions)

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(results_df)

best_row = results_df.loc[results_df["feature_set"].eq("tfidf_engineered")].iloc[0]
best_key = (best_row["feature_set"], best_row["model"])
best_model, x_test, y_test, predictions = trained_models[best_key]

print("Selected model:", best_key)
print(classification_report(
    y_test,
    predictions,
    labels=list(range(len(label_encoder.classes_))),
    target_names=label_encoder.classes_,
    zero_division=0,
))

display(pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=list(range(len(label_encoder.classes_)))),
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
))


E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\backend\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\backend\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\backend\.venv\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,feature_set,model,accuracy,macro_f1,latency_ms_per_prediction
5,engineered_only,RandomForestClassifier,0.9905,0.876167,0.628881
6,tfidf_engineered,LogisticRegression,0.9680,0.867278,32.361989
3,engineered_only,LogisticRegression,0.9625,0.825309,2.988405
4,engineered_only,LinearSVC,0.9600,0.669379,0.029025
7,tfidf_engineered,LinearSVC,0.9180,0.574272,22.382232
0,tfidf_only,LogisticRegression,0.7735,0.458429,0.327601
8,tfidf_engineered,RandomForestClassifier,0.8835,0.440893,40.129991
1,tfidf_only,LinearSVC,0.7865,0.416219,0.039051
2,tfidf_only,RandomForestClassifier,0.8080,0.350137,36.996821


Selected model: ('tfidf_engineered', 'LogisticRegression')
              precision    recall  f1-score   support

        high       0.88      0.89      0.89       102
         low       0.99      0.98      0.99      1474
      normal       0.92      0.94      0.93       417
      urgent       0.80      0.57      0.67         7

    accuracy                           0.97      2000
   macro avg       0.90      0.85      0.87      2000
weighted avg       0.97      0.97      0.97      2000



,high,low,normal,urgent
high,91,0,10,1
low,0,1451,23,0
normal,9,18,390,0
urgent,3,0,0,4


## 8. Error Analysis

In [8]:
error_cases = sample_df.iloc[getattr(x_test, "indices", np.arange(min(len(sample_df), len(y_test))))[:0]].copy()
comparison_df = pd.DataFrame(
    {
        "actual": label_encoder.inverse_transform(y_test),
        "predicted": label_encoder.inverse_transform(predictions),
    }
)
comparison_df["is_correct"] = comparison_df["actual"].eq(comparison_df["predicted"])

print("Prediction correctness summary")
display(comparison_df["is_correct"].value_counts().rename_axis("correct").reset_index(name="count"))
print("Important limitation: labels are weak supervision from rules, not human annotations.")


Prediction correctness summary


,correct,count
0,True,1936
1,False,64


Important limitation: labels are weak supervision from rules, not human annotations.


## 9. Export Artifacts

In [9]:
import json
import joblib

artifacts_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, artifacts_dir / "priority_model.joblib")
joblib.dump(vectorizer, artifacts_dir / "vectorizer.joblib")
joblib.dump(label_encoder, artifacts_dir / "label_encoder.joblib")

metadata = {
    "selected_feature_set": best_key[0],
    "selected_model": best_key[1],
    "rows": int(len(sample_df)),
    "label_distribution": sample_df["priority_label"].value_counts().to_dict(),
    "model_comparison": results_df.to_dict(orient="records"),
    "recommendation": (
        "Use the trained ML model for high-volume routing because latency and cost are much lower; "
        "use the LLM for low-confidence or high-risk cases where reasoning is valuable."
    ),
}
(artifacts_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Exported artifacts to:", artifacts_dir)
print(json.dumps(metadata, indent=2)[:1500])


Exported artifacts to: E:\Extra Courses\AIE Bootcamp\Week 3\Project 3\decision-intelligence-assistant\artifacts
{
  "selected_feature_set": "tfidf_engineered",
  "selected_model": "LogisticRegression",
  "rows": 10000,
  "label_distribution": {
    "low": 7370,
    "normal": 2082,
    "high": 511,
    "urgent": 37
  },
  "model_comparison": [
    {
      "feature_set": "engineered_only",
      "model": "RandomForestClassifier",
      "accuracy": 0.9905,
      "macro_f1": 0.8761667362096944,
      "latency_ms_per_prediction": 0.6288807000000816
    },
    {
      "feature_set": "tfidf_engineered",
      "model": "LogisticRegression",
      "accuracy": 0.968,
      "macro_f1": 0.867277902682915,
      "latency_ms_per_prediction": 32.36198869999862
    },
    {
      "feature_set": "engineered_only",
      "model": "LogisticRegression",
      "accuracy": 0.9625,
      "macro_f1": 0.825309458594968,
      "latency_ms_per_prediction": 2.9884046000006492
    },
    {
      "feature_set": "en